# 🚀 Edge-Inference-Project: Master Orchestrator

Use this notebook to run the full ML compression pipeline and high-performance C++ server on Google Colab.

### ⏱️ Estimated Runtime (T4 GPU)
| Stage | Process | Est. Time |
| :--- | :--- | :--- |
| **1** | Setup & Imports | 1-2 mins |
| **2** | Pruning & Teacher Loading | 1 min |
| **3** | Knowledge Distillation (15 epochs) | 10-15 mins |
| **4** | INT8 Quantization & ONNX Export | 2 mins |
| **5** | C++ Server Build & Benchmark | 2 mins |
| **Total** | | **~15-22 mins** |

## 🛠️ Step 1: Environment Setup
Clone the repo and install dependencies.

In [ ]:
# @title 1. Initial Clone & Install
import os
REPO_URL = "https://github.com/JakkalaNagaVamsiKrishna/edge-inference-project.git"
REPO_NAME = "edge-inference-project"

if not os.path.exists(REPO_NAME):
    !git clone {REPO_URL}
else:
    print("Repo already exists. Use the 'Update Code' cell below to pull changes.")

%cd {REPO_NAME}

print("Installing dependencies (this may take a minute)...")
!grep -v onnxruntime requirements.txt | pip install -r /dev/stdin --quiet
!pip install onnxruntime-gpu --quiet

print("✅ Setup Complete")

In [ ]:
# @title 🔄 Update Code (Git Pull)
# Use this cell whenever a fix is pushed to GitHub to sync Colab instantly.
!git pull

## 🧪 Step 2: Run Compression Pipeline
This executes **Pruning**, **Distillation**, and **Quantization**. 

*Note: The first run will download ResNet-50 weights (~100MB) and CIFAR-10 (~170MB).*

In [ ]:
# @title 2. Start ML Pipeline (~15 mins)
!python3 -m compressor.pipeline

## ⚡ Step 3: Build & Run C++ Inference Server
We compile the high-performance C++ code using CMake and run it in the background.

In [ ]:
# @title 3. Compile & Start Server (~2 mins)

# Install ONNX Runtime dev headers for Ubuntu
!sudo apt-get update && sudo apt-get install -y libonnxruntime-dev --quiet

print("Compiling C++ Inference Server...")
# We define MODEL_PATH to point to the actual output of our pipeline
!mkdir -p build && cd build && cmake .. -DMODEL_PATH="../outputs/student_int8.onnx" -DONNXRUNTIME_ROOT="/usr" && make -j$(nproc)

import subprocess
import time

print("Starting Server in background...")
# Kill any existing server process first
!pkill edge_cv_server || true

server_proc = subprocess.Popen(["./build/edge_cv_server"], stdout=subprocess.PIPE, stderr=subprocess.PIPE)
time.sleep(5)  
print("🚀 Server is Live at http://localhost:8080")

## 📊 Step 4: Benchmark & Dashboard
Validate the quantized model's performance.

In [ ]:
# @title 4. Run Benchmark Runner
!python3 -m benchmark.runner --n 100

In [ ]:
# @title 5. Launch Observability Dashboard
from google.colab.output import serve_kernel_port
import subprocess

print("Starting Dashboard...")
!pkill -f "python3 -m dashboard.app" || true
subprocess.Popen(["python3", "-m", "dashboard.app"])
time.sleep(2)

print("Click the link below to open the dashboard:")
serve_kernel_port(5000)